# Evaluation Only: Vanilla Baseline (3 Projects)
## Running on Google Colab Pro (T4 GPU)

This notebook **only runs pass@k evaluation** on pre-generated vanilla completions.

**Projects evaluated here (3 of 4):**
- `searcharray` (6 tasks)
- `UHGEval` (1 task → 3 namespaces)
- `sd-webui-forge` (3 tasks)

**Skipped:** `easyvolcap` — evaluate on HPC instead (needs CUDA libs not available on Colab).

### Prerequisites
You must already have the `output_vanilla_4projects/student_posttest_vanilla/` folder with
`completion.jsonl` files from Phase 2 (code generation). Upload them to Google Drive before running.

### Strategy
Each project has different Python dependencies. To avoid conflicts, we evaluate **one project at a time**
using an **isolated virtual environment** per project (same approach as the HPC `eval_by_project.slurm`).

---
## 0. GPU Check & Configuration

In [1]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout.split('\n')[0])
    print('✅ GPU available')
else:
    print('⚠️  No GPU — change runtime to T4 if needed')

Sat May 23 17:31:25 2026       
✅ GPU available


In [2]:
# ===== CONFIGURATION =====
from google.colab import userdata
import os

HF_TOKEN = userdata.get('HF_TOKEN')

# ===== STORAGE =====
DRIVE_DIR = "/content/drive/MyDrive/Coding-Tutor-Colab"
WORK_DIR  = f"{DRIVE_DIR}/work"
DATA_DIR  = f"{DRIVE_DIR}/data"

# ===== MODEL (for path construction only — no inference in this notebook) =====
TUTOR_MODEL_ID = "meta-llama/Llama-3.1-70B-Instruct"

# ===== PIPELINE SETTINGS =====
STUDENT_LEVELS = ["low_level", "med_level", "high_level"]
MAX_INTERACTION_ROUND = 8
N_COMPLETIONS = 10
K_LIST = "1,3,5,10"

# ===== DATASET IDS =====
TUTOR_AGENTS_DATASET = "nlpscu/Tutor-Agents"

# ===== 3 PROJECTS TO EVALUATE (skip easyvolcap) =====
SELECTED_PROJECTS = {
    "searcharray":    ["searcharray"],
    "UHGEval":        ["xinhua"],
    "sd-webui-forge":  ["gfpgan_model", "codeformer_model"],
}

# Map notebook project keys → Source_Code folder names
PROJECT_FOLDER_MAP = {
    "searcharray":    "searcharray",
    "UHGEval":        "UHGEval",
    "sd-webui-forge":  "stable-diffusion-webui-forge",
}

# Validate
assert HF_TOKEN, "Add HF_TOKEN to Colab Secrets (🔑 sidebar)!"
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
print("✅ Configuration loaded.")
print(f"   Projects: {list(SELECTED_PROJECTS.keys())}")
print(f"   (easyvolcap skipped — evaluate on HPC)")

✅ Configuration loaded.
   Projects: ['searcharray', 'UHGEval', 'sd-webui-forge']
   (easyvolcap skipped — evaluate on HPC)


---
## 1. Mount Drive & Clone Repo

In [4]:
import os, shutil
from google.colab import drive

mount_point = "/content/drive"
if os.path.exists(mount_point):
    shutil.rmtree(mount_point)
drive.mount(mount_point)

Mounted at /content/drive


In [5]:
# Clone Coding-Tutor repo (for pass_k.py and utilities)
if not os.path.exists(f"{WORK_DIR}/.git"):
    !git clone https://github.com/iwangjian/Coding-Tutor.git {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull --ff-only
print(f"✅ Repo ready at {WORK_DIR}")

Already up to date.
✅ Repo ready at /content/drive/MyDrive/Coding-Tutor-Colab/work


---
## 2. Download EvoCodeBench Dataset
We need the source code + metadata for running pytest-based evaluation.

In [13]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download, login
login(token=HF_TOKEN)

# Download EvoCodeBench-2403 (contains Source_Code + metadata)
EVOBENCH_ROOT = f"{DATA_DIR}/EvoCodeBench-2403"
if not os.path.exists(EVOBENCH_ROOT):
    print("⬇️  Downloading EvoCodeBench-2403...")
    snapshot_download(
        "pkuzqh/EvoCodeBench-2403",
        local_dir=EVOBENCH_ROOT,
        repo_type="dataset",
        token=HF_TOKEN
    )
    print("✅ EvoCodeBench downloaded.")
else:
    print("✅ EvoCodeBench already exists.")

SOURCE_CODE_ROOT = f"{EVOBENCH_ROOT}/Source_Code"

# Resolve metadata file (could be data.jsonl or metadata.jsonl)
METADATA_FILE = None
for candidate in [f"{EVOBENCH_ROOT}/data.jsonl", f"{EVOBENCH_ROOT}/metadata.jsonl"]:
    if os.path.exists(candidate) and os.path.getsize(candidate) > 0:
        METADATA_FILE = candidate
        break
assert METADATA_FILE, f"No metadata file found in {EVOBENCH_ROOT}!"
print(f"✅ Metadata: {METADATA_FILE}")
print(f"✅ Source code: {SOURCE_CODE_ROOT}")

✅ EvoCodeBench already exists.
✅ Metadata: /content/drive/MyDrive/Coding-Tutor-Colab/data/EvoCodeBench-2403/metadata.jsonl
✅ Source code: /content/drive/MyDrive/Coding-Tutor-Colab/data/EvoCodeBench-2403/Source_Code


In [14]:
# ===== Diagnostic: Inspect metadata file =====
import json, os
from collections import Counter

print(f"Metadata file: {METADATA_FILE}")
print(f"File size: {os.path.getsize(METADATA_FILE) / 1024:.1f} KB")

# Read all entries and extract project names from completion_path
projects = Counter()
namespaces_by_project = {}
total = 0
with open(METADATA_FILE) as f:
    for line in f:
        if not line.strip():
            continue
        js = json.loads(line)
        total += 1
        proj = js['completion_path'].split('/')[0]
        projects[proj] += 1
        if proj not in namespaces_by_project:
            namespaces_by_project[proj] = []
        namespaces_by_project[proj].append(js['namespace'])

print(f"\nTotal entries: {total}")
print(f"\nProjects found in metadata (completion_path prefix):")
for proj, count in sorted(projects.items()):
    print(f"  {proj}: {count} tasks")

# Show which of our 3 target projects match
print(f"\n--- Matching our target projects ---")
for our_name, folder_name in PROJECT_FOLDER_MAP.items():
    count = projects.get(folder_name, 0)
    status = '✅' if count > 0 else '❌'
    print(f"  {status} {our_name} → looking for '{folder_name}/' → {count} tasks")
    if count == 0:
        # Check if a case-insensitive match exists
        for proj in projects:
            if proj.lower() == folder_name.lower():
                print(f"     ⚠️  Found case-different match: '{proj}' ({projects[proj]} tasks)")

# Also list Source_Code folders for cross-reference
print(f"\n--- Source_Code folders on disk ---")
if os.path.isdir(SOURCE_CODE_ROOT):
    for d in sorted(os.listdir(SOURCE_CODE_ROOT)):
        if os.path.isdir(os.path.join(SOURCE_CODE_ROOT, d)):
            meta_count = projects.get(d, 0)
            print(f"  {d}/ → {meta_count} tasks in metadata")
else:
    print(f"  ⚠️  {SOURCE_CODE_ROOT} not found!")

Metadata file: /content/drive/MyDrive/Coding-Tutor-Colab/data/EvoCodeBench-2403/metadata.jsonl
File size: 385.0 KB

Total entries: 275

Projects found in metadata (completion_path prefix):
  AutoRAG: 13 tasks
  EasyVolcap: 20 tasks
  Generalizable-BEV: 8 tasks
  Python-Type-Challenges: 1 tasks
  Test-Agent: 1 tasks
  UHGEval: 3 tasks
  UniRef: 23 tasks
  XAgent: 3 tasks
  camp_zipnerf: 54 tasks
  contrastors: 1 tasks
  deluder: 3 tasks
  gaussian-splatting-lightning: 1 tasks
  litdata: 59 tasks
  microagents: 18 tasks
  microsearch: 2 tasks
  nlm-ingestor: 4 tasks
  ollama-python: 12 tasks
  open-iris: 14 tasks
  openlogprobs: 1 tasks
  scepter: 1 tasks
  searcharray: 6 tasks
  skfolio: 13 tasks
  stable-diffusion-webui-forge: 3 tasks
  stable-fast: 2 tasks
  tanuki_py: 9 tasks

--- Matching our target projects ---
  ✅ searcharray → looking for 'searcharray/' → 6 tasks
  ✅ UHGEval → looking for 'UHGEval/' → 3 tasks
  ✅ sd-webui-forge → looking for 'stable-diffusion-webui-forge/' → 3 ta

---
## 3. Upload / Verify Pre-generated Completions

Your `output_vanilla_4projects/student_posttest_vanilla/` folder should already be on Google Drive.

Expected structure:
```
output_vanilla_4projects/student_posttest_vanilla/
  searcharray/vanilla/Llama-3.1-70B-Instruct/{level}/round_{N}/completion.jsonl
  UHGEval/vanilla/Llama-3.1-70B-Instruct/{level}/round_{N}/completion.jsonl
  sd-webui-forge/vanilla/Llama-3.1-70B-Instruct/{level}/round_{N}/completion.jsonl
```

If the files are somewhere else, update `POSTTEST_BASE` below.

In [15]:
# ===== Point to your pre-generated completions =====
# Option A: If you uploaded output_vanilla/ to the same Drive folder:
POSTTEST_BASE = f"{DRIVE_DIR}/output_vanilla/student_posttest_vanilla"

# Option B: If completions are in the repo's output dir from Phase 2:
# POSTTEST_BASE = f"{DRIVE_DIR}/output_vanilla/student_posttest_vanilla"

# ===== Verify completions exist =====
import json, os
model_name = TUTOR_MODEL_ID.split("/")[-1]
ROUNDS = list(range(1, MAX_INTERACTION_ROUND + 1))

print("📋 Completion files found:")
total_files = 0
for proj_name in SELECTED_PROJECTS:
    proj_files = 0
    for level in STUDENT_LEVELS:
        for rdx in ROUNDS:
            comp_file = f"{POSTTEST_BASE}/{proj_name}/vanilla/{model_name}/{level}/round_{rdx}/completion.jsonl"
            if os.path.exists(comp_file):
                with open(comp_file) as f:
                    n_tasks = sum(1 for l in f if l.strip())
                proj_files += 1
    total_files += proj_files
    status = "✅" if proj_files > 0 else "❌"
    print(f"  {status} {proj_name}: {proj_files} completion files")

if total_files == 0:
    print("\n⚠️  No completion files found! Upload your output_vanilla_4projects/ folder to Drive.")
    print(f"   Expected location: {POSTTEST_BASE}")
else:
    print(f"\n✅ Total: {total_files} completion files ready for evaluation.")

📋 Completion files found:
  ✅ searcharray: 24 completion files
  ✅ UHGEval: 8 completion files
  ✅ sd-webui-forge: 18 completion files

✅ Total: 50 completion files ready for evaluation.


---
## 4. Evaluate Each Project in Isolated venv

**Why isolated venvs?** Each project has different (and often conflicting) Python dependencies.
For example, `sd-webui-forge` pins old `transformers` while `searcharray` needs newer versions.

We create a temporary venv per project, install its deps, run pass@k, then tear it down.
This mirrors the HPC `eval_by_project.slurm` approach.

In [25]:
# ===== OPTIONAL: Clear previous (broken) test results =====
# Run this if a previous evaluation produced all-zero results
# and you want to re-evaluate from scratch.

import os, glob

model_name = TUTOR_MODEL_ID.split('/')[-1]
deleted = 0
for proj_name in SELECTED_PROJECTS:
    pattern = f"{POSTTEST_BASE}/{proj_name}/vanilla/{model_name}/**/test_results.jsonl"
    for f in glob.glob(pattern, recursive=True):
        os.remove(f)
        deleted += 1

print(f"🧹 Deleted {deleted} old test_results.jsonl files.")
print("   The evaluation cell will now re-run all tests.")

🧹 Deleted 50 old test_results.jsonl files.
   The evaluation cell will now re-run all tests.


In [22]:
# ===== DEBUG: Test one searcharray case manually =====
# This cell sets up the same venv + env as the eval loop,
# then runs a single pytest to show the FULL error output.

import subprocess, sys, os, json

project_folder = "searcharray"
venv_path = f"/content/eval_venvs/{project_folder}"
venv_python = f"{venv_path}/bin/python"
venv_bin = f"{venv_path}/bin"

# Recreate venv if needed
if not os.path.exists(venv_python):
    print("Creating venv...")
    import shutil
    if os.path.exists(venv_path):
        shutil.rmtree(venv_path)
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", venv_path], check=True)
    get_pip = "/content/eval_venvs/get-pip.py"
    if not os.path.exists(get_pip):
        import urllib.request
        urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", get_pip)
    subprocess.run([venv_python, get_pip, "--quiet"], capture_output=True, check=True)
    EVAL_DEPS = ["numpy", "tqdm", "psutil", "func_timeout", "dill", "pytest", "pytest-runner"]
    subprocess.run([venv_python, "-m", "pip", "install", "--quiet"] + EVAL_DEPS, capture_output=True)
    # Install searcharray deps
    proj_path = os.path.join(SOURCE_CODE_ROOT, project_folder)
    req = os.path.join(proj_path, "requirements.txt")
    if os.path.isfile(req):
        subprocess.run([venv_python, "-m", "pip", "install", "--quiet", "-r", req], capture_output=True)
    setup = os.path.join(proj_path, "setup.py")
    pyproject = os.path.join(proj_path, "pyproject.toml")
    if os.path.isfile(setup) or os.path.isfile(pyproject):
        subprocess.run([venv_python, "-m", "pip", "install", "--quiet", "-e", proj_path, "--no-deps"], capture_output=True)

# Build the same env as the eval loop
eval_env = os.environ.copy()
eval_env["PATH"] = f"{venv_bin}:{eval_env.get('PATH', '')}"
eval_env["VIRTUAL_ENV"] = venv_path
eval_env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
eval_env["PYTHONUNBUFFERED"] = "1"

proj_path = os.path.join(SOURCE_CODE_ROOT, project_folder)

# Step 1: Check which pytest is being used
print("=== Which pytest? ===")
r = subprocess.run(["bash", "-c", "which pytest"], env=eval_env, capture_output=True, text=True)
print(f"  pytest path: {r.stdout.strip()}")
print(f"  venv python: {venv_python}")
r = subprocess.run(["bash", "-c", "python --version"], env=eval_env, capture_output=True, text=True)
print(f"  bash python: {r.stdout.strip()}")

# Step 2: Find a test from metadata
print("\n=== Finding a test case ===")
with open(METADATA_FILE) as f:
    for line in f:
        js = json.loads(line)
        if js['completion_path'].startswith('searcharray/'):
            print(f"  namespace: {js['namespace']}")
            print(f"  completion_path: {js['completion_path']}")
            print(f"  tests: {js['tests']}")
            test_cmd = js['tests'][0]
            break

# Step 3: Try pytest --collect-only first (just checks imports)
print(f"\n=== pytest --collect-only {test_cmd} ===")
r = subprocess.run(
    ["bash", "-c", f"pytest --collect-only {test_cmd}"],
    cwd=proj_path, env=eval_env,
    capture_output=True, text=True, timeout=30
)
print(f"  return code: {r.returncode}")
if r.stdout:
    print(f"  STDOUT:\n{r.stdout[-1000:]}")
if r.stderr:
    print(f"  STDERR:\n{r.stderr[-1000:]}")

# Step 4: Actually run the test
print(f"\n=== pytest -v {test_cmd} ===")
r = subprocess.run(
    ["bash", "-c", f"pytest -v {test_cmd}"],
    cwd=proj_path, env=eval_env,
    capture_output=True, text=True, timeout=30
)
print(f"  return code: {r.returncode}")
if r.stdout:
    print(f"  STDOUT:\n{r.stdout[-2000:]}")
if r.stderr:
    print(f"  STDERR:\n{r.stderr[-1000:]}")

=== Which pytest? ===
  pytest path: /content/eval_venvs/searcharray/bin/pytest
  venv python: /content/eval_venvs/searcharray/bin/python
  bash python: Python 3.12.13

=== Finding a test case ===
  namespace: searcharray.postings.SearchArray.positions
  completion_path: searcharray/searcharray/postings.py
  tests: ['test/test_phrase_matches.py::test_positions', 'test/test_phrase_matches.py::test_positions_mask', 'test/test_phrase_matches.py::test_positions_mask_single']

=== pytest --collect-only test/test_phrase_matches.py::test_positions ===
  return code: 0
  STDOUT:
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-7.4.3, pluggy-1.3.0
benchmark: 4.0.0 (defaults: timer=time.perf_counter disable_gc=False min_rounds=5 min_time=0.000005 max_time=1.0 calibration_precision=10 warmup=False warmup_iterations=100000)
rootdir: /content/drive/MyDrive/Coding-Tutor-Colab/data/EvoCodeBench-2403/Source_Code/searcharray
confi

In [23]:
# ===== DEBUG: Simulate what pass_k.py does for one completion =====
# pass_k.py replaces the function body in the source code with the
# generated completion, runs pytest, then restores the original.
# Let's do that manually and see what happens.

import subprocess, sys, os, json, shutil, textwrap

project_folder = "searcharray"
venv_path = f"/content/eval_venvs/{project_folder}"
venv_python = f"{venv_path}/bin/python"
venv_bin = f"{venv_path}/bin"
proj_path = os.path.join(SOURCE_CODE_ROOT, project_folder)

eval_env = os.environ.copy()
eval_env["PATH"] = f"{venv_bin}:{eval_env.get('PATH', '')}"
eval_env["VIRTUAL_ENV"] = venv_path
eval_env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
eval_env["PYTHONUNBUFFERED"] = "1"

# Step 1: Find first task in metadata
print("=== Step 1: Load metadata for first searcharray task ===")
meta = None
with open(METADATA_FILE) as f:
    for line in f:
        js = json.loads(line)
        if js['completion_path'].startswith('searcharray/'):
            meta = js
            break
print(f"  namespace:       {meta['namespace']}")
print(f"  completion_path: {meta['completion_path']}")
print(f"  body_position:   {meta['body_position']}")
print(f"  indent:          {meta.get('indent', '?')}")
print(f"  tests:           {meta['tests']}")

# Step 2: Show the original function body
src_file = os.path.join(SOURCE_CODE_ROOT, meta['completion_path'])
print(f"\n=== Step 2: Original source at {meta['completion_path']} ===")
with open(src_file) as f:
    lines = f.readlines()
sos, eos = meta['body_position'][0] - 1, meta['body_position'][1]
print(f"  Body lines {sos+1}-{eos} (0-indexed {sos}-{eos}):")
for i in range(max(0, sos-2), min(len(lines), eos+2)):
    marker = '>>>' if sos <= i < eos else '   '
    print(f"  {marker} L{i+1}: {lines[i].rstrip()}")

# Step 3: Load first completion
model_name = TUTOR_MODEL_ID.split('/')[-1]
comp_file = f"{POSTTEST_BASE}/searcharray/vanilla/{model_name}/low_level/round_1/completion.jsonl"
print(f"\n=== Step 3: First completion from {comp_file} ===")
with open(comp_file) as f:
    comp_js = json.loads(f.readline())
completion_text = comp_js['completion']
print(f"  namespace: {comp_js['namespace']}")
print(f"  completion length: {len(completion_text)} chars")
print(f"  completion preview:")
for line in completion_text.split('\n')[:15]:
    print(f"    |{line}")
if completion_text.count('\n') > 15:
    print(f"    ... ({completion_text.count(chr(10))} total lines)")

# Step 4: Do the replacement (same as pass_k.py SetUp_evaluation)
print(f"\n=== Step 4: Simulating SetUp_evaluation ===")
indent = meta.get('indent', 4)
adjusted = textwrap.indent(textwrap.dedent(completion_text), ' ' * indent)
print(f"  indent: {indent}")
print(f"  adjusted completion preview:")
for line in adjusted.split('\n')[:10]:
    print(f"    |{line}")

# Backup, replace, test, restore
backup = src_file + '.debug_backup'
shutil.copy2(src_file, backup)

try:
    new_lines = lines[:sos] + ['\n', adjusted, '\n'] + lines[eos:]
    with open(src_file, 'w') as f:
        f.write(''.join(new_lines))

    # Show what the file looks like now around the replacement
    print(f"\n  Modified file around replacement:")
    with open(src_file) as f:
        mod_lines = f.readlines()
    for i in range(max(0, sos-2), min(len(mod_lines), sos+15)):
        print(f"    L{i+1}: {mod_lines[i].rstrip()}")

    # Run the test
    test_cmd = meta['tests'][0]
    print(f"\n=== Step 5: Running pytest {test_cmd} ===")
    r = subprocess.run(
        ["bash", "-c", f"pytest -v {test_cmd}"],
        cwd=proj_path, env=eval_env,
        capture_output=True, text=True, timeout=30
    )
    print(f"  return code: {r.returncode}")
    if r.stdout:
        print(f"  STDOUT:\n{r.stdout[-2000:]}")
    if r.stderr:
        print(f"  STDERR:\n{r.stderr[-1000:]}")
finally:
    # Restore original
    shutil.copy2(backup, src_file)
    os.remove(backup)
    print("\n✅ Original source restored.")

=== Step 1: Load metadata for first searcharray task ===
  namespace:       searcharray.postings.SearchArray.positions
  completion_path: searcharray/searcharray/postings.py
  body_position:   [559, 562]
  indent:          8
  tests:           ['test/test_phrase_matches.py::test_positions', 'test/test_phrase_matches.py::test_positions_mask', 'test/test_phrase_matches.py::test_positions_mask_single']

=== Step 2: Original source at searcharray/searcharray/postings.py ===
  Body lines 559-562 (0-indexed 558-562):
      L557:     def positions(self, token: str, key=None) -> List[np.ndarray]:
      L558:         """Return a list of lists of positions of the given term."""
  >>> L559:         term_id = self.term_dict.get_term_id(token)
  >>> L560:         key = self.term_mat.rows[key] if key is not None else self.term_mat.rows
  >>> L561:         posns = self.posns.positions(term_id, doc_ids=key)
  >>> L562:         return posns
      L563: 
      L564:     def and_query(self, tokens: Union

In [27]:
import re
from pathlib import Path

passk_path = Path(f"{WORK_DIR}/traver/parser/pass_k.py")

code = passk_path.read_text()

# Make sure sys is imported
if "import sys" not in code:
    code = code.replace("import subprocess\n", "import subprocess\nimport sys\n")

new_execution_tests = r'''
@func_set_timeout(120)
def execution_tests(test, project_path):
    command = [sys.executable, "-m", "pytest", "-q", test]
    process = subprocess.Popen(
        command,
        cwd=project_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
    )

    try:
        while True:
            process_id = process.pid
            process_memory = psutil.Process(process_id).memory_info().rss

            if process_memory > 5 * 1024 * 1024 * 1024:
                process.terminate()
                out, err = process.communicate(timeout=5)
                print(f"Out of Memory: {project_path}")
                print("STDOUT tail:\\n", out[-4000:] if out else "")
                print("STDERR tail:\\n", err[-4000:] if err else "")
                return False

            return_code = process.poll()

            if return_code is not None:
                out, err = process.communicate()

                if return_code != 0:
                    print(f"Execution Error with return code {return_code}: {project_path}")
                    print(f"Command: {' '.join(command)}")
                    print("STDOUT tail:\\n", out[-4000:] if out else "")
                    print("STDERR tail:\\n", err[-4000:] if err else "")
                    return False

                break

    except Exception as e:
        process.terminate()
        out, err = process.communicate()
        print(f"Other Error: {project_path}: {repr(e)}")
        print("STDOUT tail:\\n", out[-4000:] if out else "")
        print("STDERR tail:\\n", err[-4000:] if err else "")
        return False

    finally:
        if process.poll() is None:
            process.terminate()
            process.wait()

    return True
'''

pattern = r'@func_set_timeout\(\d+\)\s*def execution_tests\(test, project_path\):.*?(?=\n\ndef compute_pass_at_k)'

code_new, count = re.subn(
    pattern,
    new_execution_tests.strip(),
    code,
    flags=re.DOTALL,
)

print("Matched execution_tests blocks:", count)

if count != 1:
    raise RuntimeError("Could not uniquely patch execution_tests(). Please print that function from pass_k.py.")

passk_path.write_text(code_new)
print("✅ Patched execution_tests() successfully.")

Matched execution_tests blocks: 1
✅ Patched execution_tests() successfully.


In [29]:
import subprocess, os, sys, json, shutil, tempfile, time

model_name = TUTOR_MODEL_ID.split("/")[-1]
ROUNDS = list(range(1, MAX_INTERACTION_ROUND + 1))

VENV_BASE = "/content/eval_venvs"
os.makedirs(VENV_BASE, exist_ok=True)

# Packages needed by pass_k.py
EVAL_DEPS = ["numpy", "tqdm", "psutil", "func_timeout", "dill", "pytest", "pytest-runner"]

# Packages to skip from requirements.txt (problematic on Colab)
SKIP_PATTERNS = [
    "git+", "cuda-python", "pyopengl", "open3d", "torch-scatter",
    "flash-attn", "triton", "spconv", "pytorch3d",
]

# Version overrides for dependency conflicts
VERSION_OVERRIDES = {
    "transformers==4.30.2": "transformers>=4.30",
    "transformers==4.30.1": "transformers>=4.30",
}

# Extra deps per project (replacements for git+ deps, etc.)
PROJECT_EXTRA_DEPS = {
    "stable-diffusion-webui-forge": [
        "safetensors", "accelerate", "diffusers", "spandrel",
        "facexlib", "gfpgan", "basicsr", "lark", "gradio",
    ],
    "searcharray": [],
    "UHGEval": [],
}


def create_venv(project_folder):
    """Create an isolated venv for one project.

    Colab's Python doesn't ship ensurepip, so we create the venv
    with --without-pip and then bootstrap pip via get-pip.py.
    """
    venv_path = os.path.join(VENV_BASE, project_folder)
    if os.path.exists(venv_path):
        shutil.rmtree(venv_path)

    print(f"  🔧 Creating venv at {venv_path}...")
    subprocess.run([sys.executable, "-m", "venv", "--without-pip", venv_path], check=True)

    venv_python = os.path.join(venv_path, "bin", "python")

    # Bootstrap pip into the venv via get-pip.py
    get_pip_path = os.path.join(VENV_BASE, "get-pip.py")
    if not os.path.exists(get_pip_path):
        print(f"  ⬇️  Downloading get-pip.py...")
        import urllib.request
        urllib.request.urlretrieve("https://bootstrap.pypa.io/get-pip.py", get_pip_path)

    print(f"  📦 Bootstrapping pip...")
    subprocess.run([venv_python, get_pip_path, "--quiet"], capture_output=True, check=True)

    # Upgrade pip
    subprocess.run([venv_python, "-m", "pip", "install", "--quiet", "--upgrade", "pip"],
                   capture_output=True)

    # Install eval dependencies
    print(f"  📦 Installing eval dependencies...")
    subprocess.run([venv_python, "-m", "pip", "install", "--quiet"] + EVAL_DEPS,
                   capture_output=True)

    return venv_path, venv_python


def install_project_deps(venv_python, project_folder):
    """Install project-specific dependencies into the venv."""
    proj_path = os.path.join(SOURCE_CODE_ROOT, project_folder)
    req_file = os.path.join(proj_path, "requirements.txt")
    setup_py = os.path.join(proj_path, "setup.py")
    pyproject = os.path.join(proj_path, "pyproject.toml")

    if os.path.isfile(req_file):
        print(f"  📦 Installing {project_folder}/requirements.txt...")
        with open(req_file) as f:
            lines = [l.strip() for l in f if l.strip() and not l.strip().startswith("#")]

        safe_deps = []
        skipped = []
        for line in lines:
            if line in VERSION_OVERRIDES:
                line = VERSION_OVERRIDES[line]
            skip = any(pat.lower() in line.lower() for pat in SKIP_PATTERNS)
            if skip:
                skipped.append(line)
            else:
                safe_deps.append(line)

        if skipped:
            print(f"    ⏭️  Skipping {len(skipped)} problematic deps")

        if safe_deps:
            result = subprocess.run(
                [venv_python, "-m", "pip", "install", "--quiet"] + safe_deps,
                capture_output=True, text=True
            )
            if result.returncode != 0:
                print(f"    ⚠️  Batch install had errors, trying one-by-one...")
                for dep in safe_deps:
                    r = subprocess.run(
                        [venv_python, "-m", "pip", "install", "--quiet", dep],
                        capture_output=True, text=True
                    )
                    if r.returncode != 0:
                        print(f"    ❌ Failed: {dep}")

    # Install extra deps
    extras = PROJECT_EXTRA_DEPS.get(project_folder, [])
    if extras:
        print(f"  📦 Installing {len(extras)} extra deps...")
        for dep in extras:
            subprocess.run(
                [venv_python, "-m", "pip", "install", "--quiet", dep],
                capture_output=True, text=True
            )

    # Install project in dev mode (--no-deps)
    if os.path.isfile(setup_py) or os.path.isfile(pyproject):
        print(f"  📦 Installing {project_folder} in dev mode...")
        subprocess.run(
            [venv_python, "-m", "pip", "install", "--quiet", "-e", proj_path, "--no-deps"],
            capture_output=True, text=True
        )


def update_test_paths(project_folder):
    """Inject sys.path.append into test files so pytest finds project modules."""
    proj_path = os.path.join(SOURCE_CODE_ROOT, project_folder)
    updated = 0
    with open(METADATA_FILE) as f:
        for line in f:
            if not line.strip():
                continue
            data = json.loads(line)
            if not data["completion_path"].startswith(project_folder + "/"):
                continue
            for test in data.get("tests", []):
                test_path = test.split("::")[0]
                full_test_path = os.path.join(SOURCE_CODE_ROOT, project_folder, test_path)
                if not os.path.exists(full_test_path):
                    full_test_path = os.path.join(SOURCE_CODE_ROOT, test_path)
                if not os.path.exists(full_test_path):
                    continue
                with open(full_test_path, "r") as tf:
                    code = tf.read()
                inject = f'import sys\nsys.path.append("{proj_path}")\n'
                if inject not in code:
                    code = inject + code
                    with open(full_test_path, "w") as tf:
                        tf.write(code)
                    updated += 1
    if updated:
        print(f"  ✅ Updated {updated} test file(s) with sys.path")
    else:
        print(f"  ✅ Test paths already up to date")


def create_filtered_metadata(project_folder, output_path):
    """Create a metadata file containing only tasks for this project."""
    count = 0
    with open(METADATA_FILE) as f, open(output_path, 'w') as out:
        for line in f:
            if not line.strip():
                continue
            js = json.loads(line)
            if js['completion_path'].startswith(project_folder + '/'):
                out.write(line)
                count += 1
    print(f"  📋 {count} tasks for {project_folder} in filtered metadata")
    return count


def run_passk_for_project(proj_name, project_folder, venv_python):
    """Run pass@k evaluation for all levels × rounds of one project."""
    tested = 0
    skipped = 0

    # Create filtered metadata for this project
    filtered_data = os.path.join(VENV_BASE, project_folder, "data_filtered.jsonl")
    n_tasks = create_filtered_metadata(project_folder, filtered_data)
    if n_tasks == 0:
        print(f"  ⚠️  No tasks in metadata for {project_folder}")
        return tested, skipped

    # Create a typing_extensions Sentinel patch script.
    # searcharray (or its deps) imports typing_extensions.Sentinel which was
    # renamed to _Sentinel in newer versions. This patch runs before every
    # subprocess pytest call via PYTHONSTARTUP.
    venv_path = os.path.dirname(os.path.dirname(venv_python))  # .../eval_venvs/<proj>
    startup_file = os.path.join(venv_path, "_eval_startup.py")
    with open(startup_file, "w") as f:
        f.write("import typing_extensions as _te\n")
        f.write("if not hasattr(_te, 'Sentinel') and hasattr(_te, '_Sentinel'):\n")
        f.write("    _te.Sentinel = _te._Sentinel\n")

    # Build environment that ensures the venv's bin/ is first in PATH.
    # This is critical because pass_k.py runs `bash -c "pytest ..."` which
    # would otherwise pick up the system pytest (outside the venv).
    venv_bin = os.path.join(venv_path, "bin")
    eval_env = os.environ.copy()
    eval_env["PATH"] = f"{venv_bin}:{eval_env.get('PATH', '')}"
    eval_env["VIRTUAL_ENV"] = venv_path
    eval_env["PYTHONPATH"] = f"{WORK_DIR}/traver:{WORK_DIR}"
    eval_env["PYTHONSTARTUP"] = startup_file
    eval_env["PYTHONUNBUFFERED"] = "1"

    # Restore any backup files before testing
    subprocess.run(
        [venv_python, f"{WORK_DIR}/traver/utils/check_source_code.py", SOURCE_CODE_ROOT],
        cwd=WORK_DIR, capture_output=True, env=eval_env
    )

    for level in STUDENT_LEVELS:
        for rdx in ROUNDS:
            comp_file = f"{POSTTEST_BASE}/{proj_name}/vanilla/{model_name}/{level}/round_{rdx}/completion.jsonl"
            log_file  = f"{POSTTEST_BASE}/{proj_name}/vanilla/{model_name}/{level}/round_{rdx}/test_results.jsonl"

            if not os.path.exists(comp_file):
                continue

            with open(comp_file) as f:
                n_comp = sum(1 for l in f if l.strip())

            # Skip if already evaluated
            if os.path.exists(log_file):
                with open(log_file) as f:
                    n_tested = sum(1 for l in f if l.strip())
                if n_tested >= n_comp:
                    skipped += 1
                    continue

            print(f"  🧪 {level}/R{rdx} ({n_comp} completions)...")

            result = subprocess.run(
                [venv_python, f"{WORK_DIR}/traver/parser/pass_k.py",
                 "--output_file", comp_file,
                 "--log_file", log_file,
                 "--data_file", filtered_data,
                 "--source_code_root", SOURCE_CODE_ROOT,
                 "--k", K_LIST,
                 "--n", str(N_COMPLETIONS)],
                cwd=WORK_DIR, capture_output=True, text=True,
                env=eval_env, timeout=3600
            )

            if result.stdout:
                # Print last few lines (contains pass@k results)
                for line in result.stdout.strip().split('\n')[-5:]:
                    print(f"    {line}")
            if result.returncode != 0 and result.stderr:
                print(f"    ⚠️ Error: {result.stderr[:300]}")

            tested += 1

    return tested, skipped


# ===========================================================================
# MAIN: Evaluate each project in its own venv
# ===========================================================================
print("=" * 65)
print("  Evaluation (Isolated venv per Project)")
print("=" * 65)

grand_tested = 0
grand_skipped = 0

for proj_name, proj_folder in PROJECT_FOLDER_MAP.items():
    proj_start = time.time()

    print(f"\n{'━' * 65}")
    print(f"  📦 {proj_name} → {proj_folder}")
    print(f"{'━' * 65}")

    proj_source = os.path.join(SOURCE_CODE_ROOT, proj_folder)
    if not os.path.isdir(proj_source):
        print(f"  ⚠️  Source code not found at {proj_source}, skipping")
        continue

    # Step 1: Create isolated venv
    venv_path, venv_python = create_venv(proj_folder)

    # Step 2: Install project deps
    install_project_deps(venv_python, proj_folder)

    # Step 3: Update test paths
    update_test_paths(proj_folder)

    # Step 4: Run pass@k
    tested, skipped = run_passk_for_project(proj_name, proj_folder, venv_python)
    grand_tested += tested
    grand_skipped += skipped

    # Step 5: Cleanup venv
    elapsed = time.time() - proj_start
    print(f"  🗑️  Removing venv...")
    shutil.rmtree(venv_path, ignore_errors=True)
    print(f"  ✅ {proj_name} done ({elapsed/60:.1f} min)")

print(f"\n{'=' * 65}")
print(f"  ✅ All evaluations complete!")
print(f"     Tested: {grand_tested} | Skipped (already done): {grand_skipped}")
print(f"{'=' * 65}")

  Evaluation (Isolated venv per Project)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  📦 searcharray → searcharray
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🔧 Creating venv at /content/eval_venvs/searcharray...
  📦 Bootstrapping pip...
  📦 Installing eval dependencies...
  📦 Installing searcharray/requirements.txt...
  📦 Installing searcharray in dev mode...
  ✅ Test paths already up to date
  📋 6 tasks for searcharray in filtered metadata
  🧪 low_level/R1 (60 completions)...
     
    Pass@1: 0.0%
    Pass@3: 0.0%
    Pass@5: 0.0%
    Pass@10: 0.0%
  🧪 low_level/R2 (60 completions)...
     
    Pass@1: 0.0%
    Pass@3: 0.0%
    Pass@5: 0.0%
    Pass@10: 0.0%
  🧪 low_level/R3 (40 completions)...
     
    Pass@1: 0.0%
    Pass@3: 0.0%
    Pass@5: 0.0%
    Pass@10: 0.0%
  🧪 low_level/R4 (20 completions)...
     
    Pass@1: 0.0%
    Pass@3: 0.0%
    Pass@5: 0.0%
    Pass@10: 0.0%
  🧪 low_level/R5 (10 completions)...
     
    Pass@1: 0.0%

---
## 5. Results Analysis

In [30]:
import json, os, numpy as np
from collections import defaultdict

model_name = TUTOR_MODEL_ID.split("/")[-1]
LEVELS = ["low_level", "med_level", "high_level"]
ROUNDS = list(range(1, 9))
K_VALUES = [1, 3, 5, 10]

def compute_pass_at_k(n, c, k):
    if n - c < k:
        return 1.0
    return 1.0 - np.prod(1.0 - k / np.arange(n - c + 1, n + 1))


def analyze_project_results(proj_name, base_dir):
    results = {}
    for level in LEVELS:
        results[level] = {}
        for rdx in ROUNDS:
            test_file = f"{base_dir}/{proj_name}/vanilla/{model_name}/{level}/round_{rdx}/test_results.jsonl"
            comp_file = f"{base_dir}/{proj_name}/vanilla/{model_name}/{level}/round_{rdx}/completion.jsonl"

            if not os.path.exists(test_file) or not os.path.exists(comp_file):
                continue

            passed_strings = defaultdict(set)
            with open(test_file) as f:
                for line in f:
                    if line.strip():
                        js = json.loads(line)
                        if js.get('Result') == 'Pass':
                            passed_strings[js['namespace']].add(js['completion'])

            ns_total = defaultdict(int)
            ns_passed = defaultdict(int)
            with open(comp_file) as f:
                for line in f:
                    if line.strip():
                        js = json.loads(line)
                        ns = js['namespace']
                        ns_total[ns] += 1
                        if js['completion'] in passed_strings.get(ns, set()):
                            ns_passed[ns] += 1

            metrics = {}
            for k in K_VALUES:
                vals = []
                for ns, n_actual in ns_total.items():
                    c = ns_passed.get(ns, 0)
                    if n_actual >= k:
                        vals.append(compute_pass_at_k(n_actual, c, k))
                metrics[k] = np.mean(vals) * 100 if vals else 0.0

            results[level][rdx] = {
                'metrics': metrics,
                'n_tasks': len(ns_total),
                'n_passed_tasks': len(passed_strings),
            }
    return results


# Paper reference results for comparison
PAPER_RESULTS = {
    "searcharray": {
        "low_level": {"pass@1": 3.3, "pass@10": 16.7},
        "med_level": {"pass@1": 5.0, "pass@10": 16.7},
        "high_level": {"pass@1": 0.0, "pass@10": 0.0},
    },
    "UHGEval": {
        "low_level": {"pass@1": 20.0, "pass@10": 100.0},
        "med_level": {"pass@1": 40.0, "pass@10": 100.0},
        "high_level": {"pass@1": 30.0, "pass@10": 100.0},
    },
    "sd-webui-forge": {
        "low_level": {"pass@1": 43.3, "pass@10": 100.0},
        "med_level": {"pass@1": 83.3, "pass@10": 100.0},
        "high_level": {"pass@1": 40.0, "pass@10": 100.0},
    },
}

print("=" * 90)
print("  VANILLA BASELINE RESULTS — 3 PROJECTS (excl. easyvolcap)")
print("=" * 90)

all_results = {}
for proj_name in SELECTED_PROJECTS:
    results = analyze_project_results(proj_name, POSTTEST_BASE)
    all_results[proj_name] = results

    if not any(results[l] for l in LEVELS):
        print(f"\n⚠️  No results for {proj_name} (run evaluation first)")
        continue

    print(f"\n{'='*70}")
    print(f"  {proj_name}")
    print(f"{'='*70}")

    for level in LEVELS:
        if not results[level]:
            continue
        print(f"\n  --- {level} ---")
        header = f"  {'Round':<8}"
        for k in K_VALUES:
            header += f"{'P@'+str(k):<10}"
        header += f"{'Tasks':<8}{'Passed':<8}"
        print(header)
        print("  " + "-" * (len(header) - 2))

        for rdx in ROUNDS:
            if rdx not in results[level]:
                continue
            r = results[level][rdx]
            row = f"  R{rdx:<7}"
            for k in K_VALUES:
                row += f"{r['metrics'].get(k, 0):<10.1f}"
            row += f"{r['n_tasks']:<8}{r['n_passed_tasks']:<8}"
            print(row)

        if proj_name in PAPER_RESULTS and level in PAPER_RESULTS[proj_name]:
            paper = PAPER_RESULTS[proj_name][level]
            best_round = max(results[level].keys(),
                           key=lambda r: results[level][r]['metrics'].get(1, 0))
            best = results[level][best_round]
            print(f"\n  Best R{best_round}: P@1={best['metrics'][1]:.1f}% (paper: {paper['pass@1']}%)  "
                  f"P@10={best['metrics'][10]:.1f}% (paper: {paper['pass@10']}%)")

# Summary table
print(f"\n\n{'='*90}")
print("  SUMMARY: Best Round per Project × Level (vs Paper)")
print(f"{'='*90}")
print(f"  {'Project':<25}{'Level':<15}{'Best R':<8}{'Our P@1':<10}{'Paper P@1':<12}{'Our P@10':<10}{'Paper P@10':<12}")
print("  " + "-" * 85)

for proj_name in SELECTED_PROJECTS:
    results = all_results[proj_name]
    for level in LEVELS:
        if not results[level]:
            continue
        best_round = max(results[level].keys(),
                       key=lambda r: results[level][r]['metrics'].get(1, 0))
        best = results[level][best_round]
        paper = PAPER_RESULTS.get(proj_name, {}).get(level, {})
        print(f"  {proj_name:<25}{level:<15}R{best_round:<7}"
              f"{best['metrics'][1]:<10.1f}{paper.get('pass@1', 'N/A'):<12}"
              f"{best['metrics'][10]:<10.1f}{paper.get('pass@10', 'N/A'):<12}")

print(f"\n  Note: easyvolcap results should be combined from HPC evaluation.")

  VANILLA BASELINE RESULTS — 3 PROJECTS (excl. easyvolcap)

  searcharray

  --- low_level ---
  Round   P@1       P@3       P@5       P@10      Tasks   Passed  
  ----------------------------------------------------------------
  R1      0.0       0.0       0.0       0.0       6       0       
  R2      0.0       0.0       0.0       0.0       6       0       
  R3      0.0       0.0       0.0       0.0       4       0       
  R4      0.0       0.0       0.0       0.0       2       0       
  R5      0.0       0.0       0.0       0.0       1       0       
  R6      0.0       0.0       0.0       0.0       1       0       
  R7      0.0       0.0       0.0       0.0       1       0       
  R8      0.0       0.0       0.0       0.0       1       0       

  Best R1: P@1=0.0% (paper: 3.3%)  P@10=0.0% (paper: 16.7%)

  --- med_level ---
  Round   P@1       P@3       P@5       P@10      Tasks   Passed  
  ----------------------------------------------------------------
  R1      1.7      

---
## 6. Export Results to JSON
Save structured results for later combination with easyvolcap (from HPC).

In [31]:
import json

export = {}
for proj_name in SELECTED_PROJECTS:
    results = all_results.get(proj_name, {})
    export[proj_name] = {}
    for level in LEVELS:
        if not results.get(level):
            continue
        export[proj_name][level] = {}
        for rdx, r in results[level].items():
            export[proj_name][level][f"round_{rdx}"] = {
                "pass_at_k": {f"pass@{k}": round(v, 2) for k, v in r['metrics'].items()},
                "n_tasks": r['n_tasks'],
                "n_passed_tasks": r['n_passed_tasks'],
            }

export_file = f"{POSTTEST_BASE}/eval_results_3projects.json"
with open(export_file, 'w') as f:
    json.dump(export, f, indent=2)
print(f"✅ Results exported to {export_file}")
print("   Combine with easyvolcap HPC results for the full 4-project analysis.")

✅ Results exported to /content/drive/MyDrive/Coding-Tutor-Colab/output_vanilla/student_posttest_vanilla/eval_results_3projects.json
   Combine with easyvolcap HPC results for the full 4-project analysis.


---
## 📋 Summary

This notebook evaluated the vanilla baseline for **3 projects** on Colab:
- `searcharray` — 6 tasks
- `UHGEval` — 1 task (3 namespaces)
- `sd-webui-forge` — 3 tasks

**easyvolcap** (12 tasks) must be evaluated on HPC due to CUDA/GPU library requirements.

Results are saved to `eval_results_3projects.json` for combination with HPC results.